# Урок 11 - Протокол контексту моделі (MCP)

The **Протокол контексту моделі (MCP)** is an open standard that enables agents to dynamically discover and use tools, resources, and data sources at runtime. Instead of hardcoding tools into an agent, MCP lets agents connect to external servers that expose capabilities on demand.

In this lesson, you'll learn:
- Що таке MCP і чому це важливо для агентних систем
- Як працює клієнт-серверна архітектура MCP
- Як створювати агентів, які використовують виявлення інструментів у стилі MCP


## Налаштування

**Передумови:**
- Проєкт Azure AI Foundry з розгорнутою моделлю
- Виконайте `az login` для автентифікації


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import asyncio
from typing import Annotated
from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

## Що таке протокол Model Context (MCP)?

MCP визначає стандартний спосіб, за допомогою якого агенти ШІ можуть знаходити та взаємодіяти з зовнішніми інструментами та джерелами даних:

- **MCP Server**: Надає інструменти, ресурси та підказки через стандартний протокол
- **MCP Client**: Середовище виконання агента, яке підключається до серверів і виявляє доступні можливості
- **Dynamic Discovery**: Агентам не потрібні жорстко вбудовані інструменти — вони виявляють, що доступно під час виконання

Це дає змогу створювати розширювані системи агентів, у яких нові можливості можна додавати без зміни коду агента.


## Як працює MCP

```
┌─────────────┐     discover      ┌─────────────────┐
│  MCP Client  │ ──────────────► │   MCP Server     │
│  (Agent)     │                  │  (Tool Provider) │
│              │ ◄────────────── │                   │
│              │   tool results   │  • list_tools()  │
│              │                  │  • call_tool()   │
└─────────────┘                  │  • resources     │
                                  └─────────────────┘
```

1. Агент (MCP клієнт) підключається до MCP сервера
2. Сервер відповідає списком доступних інструментів та їхніми схемами
3. Потім агент може викликати будь-який виявлений інструмент у процесі міркування
4. Результати повертаються через той самий протокол


## Імітація виявлення інструментів MCP

Оскільки реальний MCP-сервер вимагає працюючого серверного процесу, ми продемонструємо цей підхід, використовуючи функції `@tool`, які моделюють те, що надав би сервіс розміщення, підключений до MCP.

У виробничому середовищі ці інструменти були б виявлені динамічно з MCP-сервера, а не визначені локально.


In [ ]:
@tool(approval_mode="never_require")
def search_accommodations(
    location: Annotated[str, "The city to search for accommodations"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"] = 2
) -> str:
    """Search for accommodations (simulating an MCP-connected Airbnb tool).
    In production, this would be discovered via MCP from an accommodation service."""
    listings = {
        "Tokyo": [
            {"name": "Shinjuku Modern Apartment", "price": 120, "rating": 4.8},
            {"name": "Traditional Ryokan in Asakusa", "price": 200, "rating": 4.9},
            {"name": "Shibuya Studio", "price": 85, "rating": 4.5},
        ],
        "Paris": [
            {"name": "Le Marais Charming Flat", "price": 150, "rating": 4.7},
            {"name": "Montmartre Artist Loft", "price": 110, "rating": 4.6},
        ],
        "Barcelona": [
            {"name": "Gothic Quarter Penthouse", "price": 130, "rating": 4.8},
            {"name": "Barceloneta Beach Flat", "price": 95, "rating": 4.4},
        ],
    }
    results = listings.get(location, [])
    if not results:
        return f"No accommodations found in {location}"
    output = f"Accommodations in {location} ({check_in} to {check_out}, {guests} guests):\n"
    for listing in results:
        output += f"  - {listing['name']}: ${listing['price']}/night (★{listing['rating']})\n"
    return output


@tool(approval_mode="never_require")
def get_local_experiences(
    location: Annotated[str, "The city to find experiences in"],
    interest: Annotated[str, "Type of experience (food, culture, adventure, etc.)"] = "all"
) -> str:
    """Get local experiences and activities (simulating an MCP-connected tourism tool)."""
    experiences = {
        "Tokyo": {
            "food": ["Tsukiji Market Tour ($45)", "Ramen Making Class ($60)", "Sake Tasting ($35)"],
            "culture": ["Tea Ceremony ($50)", "Samurai Museum ($15)", "Sumo Tournament ($80)"],
            "adventure": ["Mt. Fuji Day Trip ($120)", "Go-kart City Tour ($80)"],
        },
        "Paris": {
            "food": ["Wine & Cheese Tasting ($55)", "Cooking Class ($90)", "Market Tour ($40)"],
            "culture": ["Louvre Guided Tour ($35)", "Montmartre Art Walk ($25)"],
        },
    }
    city_exp = experiences.get(location, {})
    if not city_exp:
        return f"No experiences found in {location}"
    if interest != "all" and interest in city_exp:
        items = city_exp[interest]
        return f"{interest.title()} experiences in {location}:\n" + "\n".join(f"  - {e}" for e in items)
    output = f"All experiences in {location}:\n"
    for cat, items in city_exp.items():
        output += f"\n  {cat.title()}:\n"
        for item in items:
            output += f"    - {item}\n"
    return output

## Створення агента з інструментами у стилі MCP


In [ ]:
agent = await provider.create_agent(
    tools=[search_accommodations, get_local_experiences],
    name="AccommodationAgent",
    instructions="""You are an accommodation and travel experiences specialist powered by MCP-connected services.

Help travelers find the perfect place to stay and things to do. When searching:
1. Use the search_accommodations tool to find listings
2. Use the get_local_experiences tool to suggest activities
3. Compare options and make personalized recommendations
4. Consider the traveler's budget, interests, and travel style""",
)

response = await agent.run(
    "I'm visiting Tokyo for 5 nights in April with my partner. We love traditional Japanese culture and food. "
    "Find us a place to stay and suggest some experiences.",
    )
print(response)

## MCP у виробничому середовищі

У виробничому середовищі MCP дає змогу впроваджувати потужні шаблони:

- **Динамічне виявлення інструментів**: Агенти підключаються до серверів MCP і виявляють інструменти під час виконання
- **Слабозв'язана архітектура**: Постачальники інструментів можуть оновлюватися незалежно від агента
- **Обмін між організаціями**: Команди можуть надавати можливості через сервери MCP, які будь-який агент може використовувати
- **Підтримка Microsoft Agent Framework**: MAF має вбудовану підтримку клієнта MCP через інтеграцію `mcp`

Щоб використовувати реальний сервер MCP з MAF, ви підключаєтеся через `hosted_mcp_tool()` або інтеграцію клієнта MCP.

**Дізнайтеся більше:**
- [Специфікація MCP](https://modelcontextprotocol.io/)
- [Підтримка MCP у Microsoft Agent Framework](https://github.com/microsoft/agent-framework/tree/main/python/samples/02-agents/mcp)


## Підсумок

У цьому уроці ви дізналися:
- **MCP** — відкритий стандарт для динамічного виявлення інструментів між агентами та постачальниками інструментів
- **клієнт-серверна архітектура** дозволяє агентам виявляти можливості під час виконання
- MCP забезпечує **розширювані, слабо зв'язані системи агентів**, де інструменти можна додавати без змін у коді
- Microsoft Agent Framework надає **вбудовану підтримку MCP** для виробничого використання


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
Відмова від відповідальності:
Цей документ було перекладено із використанням сервісу машинного перекладу [Co-op Translator](https://github.com/Azure/co-op-translator). Хоча ми прагнемо до точності, просимо врахувати, що автоматичні переклади можуть містити помилки або неточності. Оригінальний документ у своїй оригінальній мові слід вважати авторитетним джерелом. Для критично важливої інформації рекомендується звернутися до професійного перекладача. Ми не несемо відповідальності за будь-які непорозуміння або неправильні тлумачення, що виникли внаслідок використання цього перекладу.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
